# Chunking inspection

This notebook runs the current PDF extractor, LLM section annotator (when OpenRouter is configured), and repository chunker on the first four pages of the sample PDF. It displays the section path attached to every chunk so the hierarchy used by retrieval is visible. It does not connect to the database or write application files. Without an OpenRouter key, it reports and uses the deterministic heading fallback.

The chunker still groups complete source paragraphs under its token limits; the model labels document structure rather than rewriting source text. Change `PAGE_LIMIT` below to inspect more pages. After editing repository code, restart the kernel and run all cells to load the new implementation.

In [1]:
from __future__ import annotations

import sys
from html import escape
from pathlib import Path
from uuid import NAMESPACE_URL, uuid5

from IPython.display import HTML, display

sys.dont_write_bytecode = True


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "rag_assistant"
        ).is_dir():
            return candidate
    raise RuntimeError("Run this notebook from somewhere inside the repository.")


REPOSITORY_ROOT = find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

In [2]:
from types import SimpleNamespace

from rag_assistant.core.config import Settings
from rag_assistant.extraction import PdfExtractor
from rag_assistant.indexing.chunking import (
    CHUNKING_VERSION,
    ChunkReference,
    ParagraphChunker,
    TiktokenTokenCounter,
)
from rag_assistant.indexing.service import DocumentIndexingService
from rag_assistant.indexing.structure import (
    OpenRouterStructureAnnotator,
    heuristic_heading_levels,
)

PAGE_LIMIT = 4
if not isinstance(PAGE_LIMIT, int) or PAGE_LIMIT < 1:
    raise ValueError("PAGE_LIMIT must be a positive integer.")
PDF_PATH = REPOSITORY_ROOT / "tests" / "fixtures" / "documents" / "sample_doc.pdf"
settings = Settings(_env_file=REPOSITORY_ROOT / ".env")
EMBEDDING_MODEL = settings.embedding_model

print(f"PDF: {PDF_PATH.name}")
print(f"Physical pages selected: 1-{PAGE_LIMIT}")
print(f"Tokenizer model: {EMBEDDING_MODEL}")
print(f"Structure annotation enabled: {settings.structure_annotation_enabled}")

PDF: sample_doc.pdf
Physical pages selected: 1-4
Tokenizer model: openai/text-embedding-3-large
Structure annotation enabled: True


In [3]:
extracted = PdfExtractor().extract(PDF_PATH.read_bytes())
selected_source_blocks = [
    block for block in extracted.paragraphs if block.page_number <= PAGE_LIMIT
]

# Deterministic temporary IDs make the production hierarchy builder inspectable without a DB.
source_paragraphs = [
    SimpleNamespace(
        id=uuid5(NAMESPACE_URL, f"{PDF_PATH.name}:{block.page_number}:{block.paragraph_number}"),
        page_number=block.page_number,
        paragraph_number=block.paragraph_number,
        kind=block.kind,
        original_text=block.original_text,
        normalized_text=block.normalized_text,
        paragraph_metadata=block.metadata,
    )
    for block in selected_source_blocks
]
annotations = heuristic_heading_levels(source_paragraphs)
annotation_mode = "heuristic fallback (OpenRouter key unavailable or annotation disabled)"
if settings.structure_annotation_enabled and settings.openrouter_api_key is not None:
    llm_annotations = await OpenRouterStructureAnnotator.from_settings(settings).annotate(
        source_paragraphs
    )
    annotations.update(llm_annotations)
    annotation_mode = f"OpenRouter model: {settings.chat_model}"
paragraphs = DocumentIndexingService._chunking_paragraphs(source_paragraphs, annotations)

headings = [
    f"level {paragraph.heading_level}: {paragraph.normalized_text}"
    for paragraph in paragraphs
    if paragraph.heading_level is not None
]
print(f"Extracted {len(paragraphs)} source paragraphs from {PAGE_LIMIT} pages.")
print(f"Annotation mode: {annotation_mode}")
print(f"Assigned {len(headings)} section headings:")
for heading in headings:
    print(f"  {heading}")

Extracted 39 source paragraphs from 4 pages.
Annotation mode: OpenRouter model: openai/gpt-6-luna
Assigned 10 section headings:
  level 1: 1 . مقدمه و پیشینه
  level 1: 2. اجزای اصلی یک سیستم RAG
  level 2: 1.2 پردازش و بارگذاری اسناد (Document Ingestion)
  level 2: 2.2 تقسیمبندی متن (Chunking)
  level 3: الف) تقسیمبندی با اندازه ثابت (Fixed-size Chunking)
  level 3: ب) تقسیمبندی معنایی (Semantic Chunking)
  level 3: ج) تقسیمبندی با همپوشانی (Overlapping Chunks)
  level 2: 3.2 تبدیل به بردار (Embedding)
  level 2: 2.4 پایگاه داده برداری (Vector Store)
  level 2: 2.5 بازیابی و رتبهبندی (Retrieval & Reranking)


In [4]:
# No chunking rules are reproduced here: this calls the current repository implementation
# with its current default target, maximum, and overlap settings.
chunker = ParagraphChunker(TiktokenTokenCounter(EMBEDDING_MODEL))
chunks = chunker.chunk(paragraphs)

settings = chunks[0].metadata
print(f"Created {len(chunks)} chunks.")
print(f"Method: {CHUNKING_VERSION}")
print(
    "Settings: "
    f"target={settings['target_tokens']} tokens, "
    f"maximum={settings['max_tokens']} tokens, "
    f"paragraph overlap={settings['paragraph_overlap']}"
)

Created 12 chunks.
Method: structure-aware-v3
Settings: target=400 tokens, maximum=600 tokens, paragraph overlap=1


## Resulting chunks

Each card shows the section path assigned by structure annotation, the complete normalized text sent for embedding, and the source citation units represented by that chunk. This makes it possible to check both token grouping and whether sibling subsections share the intended parent. Original extracted line breaks remain in each source block's `original_text`; offsets appear when an oversized source paragraph is split.

In [5]:
def reference_label(reference: ChunkReference) -> str:
    label = f"page {reference.page_number}, paragraph {reference.paragraph_number}"
    if reference.start_offset is not None:
        label += f" [{reference.start_offset}:{reference.end_offset})"
    return label


card_style = "border:1px solid #d0d7de;border-radius:8px;padding:12px 16px;margin:12px 0"
text_style = (
    "text-align:right;line-height:1.9;background:#f6f8fa;color:#1f2328;"
    "padding:12px;border-radius:6px"
)

for chunk in chunks:
    sources = "; ".join(reference_label(reference) for reference in chunk.references)
    section_path = (
        " › ".join(str(item["title"]) for item in chunk.metadata.get("section_path", []))
        or "(no section heading)"
    )
    target = int(chunk.metadata["target_tokens"])
    size_note = "above target, within hard limit" if chunk.token_count > target else "within target"
    rendered_text = escape(chunk.normalized_text).replace("\n", "<br>")
    display(
        HTML(
            f"""
            <section style="{card_style}">
              <h3 style="margin:0 0 6px">Chunk {chunk.ordinal}</h3>
              <div><strong>Section path:</strong> {escape(section_path)}</div>
              <div><strong>{chunk.token_count} tokens</strong> — {size_note}</div>
              <div style="margin:4px 0 12px"><strong>Sources:</strong> {escape(sources)}</div>
              <div dir="rtl" style="{text_style}">
                {rendered_text}
              </div>
            </section>
            """
        )
    )